# Fish Speech S2 Pro - Voice Cloning

This notebook demonstrates zero-shot voice cloning using Fish Speech S2 Pro (~4.4B parameters). It requires an A100 GPU for optimal performance due to its large parameter count and Dual-AR architecture. Expected runtime is approximately 15 minutes.

In [ ]:
!nvidia-smi

In [ ]:
# Fish Audio installs S2 from a source checkout. The "fish-speech" project on PyPI is
# not published by Fish Audio, so fetch the reviewed upstream commit by its full SHA.
!apt-get -qq update && apt-get -qq install -y portaudio19-dev libsox-dev ffmpeg
!git init -q fish-speech
!git -C fish-speech fetch -q --depth 1 https://github.com/fishaudio/fish-speech.git bdd39477922a89b904c6f2c7397f2a9bbcc60a0c
!git -C fish-speech checkout -q --detach FETCH_HEAD
!pip install -e ./fish-speech

In [ ]:
!huggingface-cli download fishaudio/s2-pro --local-dir checkpoints/s2-pro

In [ ]:
from google.colab import files

print('Please upload your 15s reference WAV file:')
uploaded = files.upload()
ref_wav_path = next(iter(uploaded))

# S2 clones from the reference audio together with its exact transcript.
ref_text = 'Replace this with the exact words spoken in your reference clip.'

In [ ]:
import subprocess
import sys

text = 'You are absolutely right. Your Claude Code session could sound like me.'

print('Loading Fish Speech S2 Pro and synthesizing speech from reference...')
subprocess.run(
    [
        sys.executable,
        'fish-speech/fish_speech/models/text2semantic/inference.py',
        '--text', text,
        '--prompt-text', ref_text,
        '--prompt-audio', ref_wav_path,
        '--checkpoint-path', 'checkpoints/s2-pro',
        '--output', 'output.wav',
    ],
    check=True,
)
print('Output saved to output.wav')

In [ ]:
from google.colab import files

files.download('output.wav')

### Next steps for the afterwords project

Because this 4.4B parameter model is too heavy for comfortable, responsive local inference on a 32GB M5 Mac, you cannot load it directly within the `afterwords` local backend environment. Instead, you should wrap the generation logic from this notebook into a lightweight FastAPI service hosted on a cloud provider (e.g., RunPod, Modal, or AWS EC2) using an A100 GPU. Then, create a `FishSpeechCloudBackend` class in your `afterwords` project that implements the `synthesize()` method by making an HTTP POST request to your cloud endpoint, transmitting the reference audio and text, and receiving the synthesized WAV bytes in return.